#Camada Gold e Views Analíticas - Arquitetura Medalhão

## Objetivo
O objetivo desta etapa é consolidar as informações de negócio processadas nas camadas anteriores (Bronze e Silver) para criar a **Camada Gold**. Esta camada é otimizada para leitura, relatórios de BI (Power BI) e dashboards executivos.

## Estrutura do Projeto
A atividade foi dividida em mini-projetos focados em áreas de negócio:
1.  **Logística:** Análise de vendas por localização e atrasos.
2.  **Comercial:** Análise temporal de vendas e indicadores financeiros.
3.  **Produto/Fashion:** Performance de categorias específicas.

In [0]:

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import DecimalType

catalogo = "medalhao"
bronze_db = "bronze"
silver_db = "silver"
gold_db = "gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalogo}.{gold_db}")
print(f"Schema {catalogo}.{gold_db} criado/verificado com sucesso.\n")

Schema medalhao.gold criado/verificado com sucesso.



In [0]:
def salvar_tabela_gold(df, nome_tabela):
    (
        df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{gold_db}.{nome_tabela}")
    )
    print(f"Tabela {gold_db}.{nome_tabela} criada com sucesso!\n")


## 1. Projeto Logística: Vendas por Localidade

A equipe de logística precisa identificar a concentração de vendas por cidade e estado para otimizar rotas de entrega.

#### gold.ft_vendas_consumidor_local
Criamos uma tabela fato desnormalizada que une os pedidos (`silver.ft_pedido_total`) com a localização dos consumidores (`silver.ft_consumidores`).

#### gold.view_total_compras_por_consumidor
Esta view consolida os dados, somando o valor total vendido e contando a quantidade de pedidos agrupados por Cidade e Estado.

In [0]:
df_pedidos_total = spark.read.table("medalhao.silver.ft_pedido_total")
df_consumidores = spark.read.table("medalhao.silver.ft_consumidores")

df_vendas_localidade = df_pedidos_total.alias("p").join(
    df_consumidores.alias("c"),
    F.col("p.id_consumidor") == F.col("c.id_consumidor"),
    "inner"
)

df_vendas_localidade = df_vendas_localidade.select(
    F.col("p.id_pedido"),
    F.col("c.id_consumidor"),
    F.col("p.valor_total_pago_brl").cast(DecimalType(12, 2)).alias("valor_total_pedido_brl"),
    F.col("c.cidade"),
    F.col("c.estado"),
    F.col("p.data_pedido").alias("data_pedido")
)

salvar_tabela_gold(df_vendas_localidade, "ft_vendas_consumidor_local")
print("Schema da tabela:")
df_vendas_localidade.printSchema()


Tabela gold.ft_vendas_consumidor_local criada com sucesso!

Schema da tabela:
root
 |-- id_pedido: string (nullable = true)
 |-- id_consumidor: string (nullable = true)
 |-- valor_total_pedido_brl: decimal(12,2) (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_pedido: date (nullable = true)



In [0]:
%sql
CREATE OR REPLACE VIEW medalhao.gold.view_total_compras_por_consumidor AS
SELECT
    cidade,
    estado,
    COUNT(DISTINCT id_pedido) AS quantidade_vendas,
    CAST(SUM(valor_total_pedido_brl) AS DECIMAL(12,2)) AS valor_total_localidade
FROM medalhao.gold.ft_vendas_consumidor_local
GROUP BY cidade, estado
ORDER BY valor_total_localidade DESC;


In [0]:
display(spark.sql("SELECT * FROM medalhao.gold.view_total_compras_por_consumidor LIMIT 5"))


cidade,estado,quantidade_vendas,valor_total_localidade
SAO PAULO,SP,15540,2203373.09
RIO DE JANEIRO,RJ,6882,1161927.36
BELO HORIZONTE,MG,2773,421765.12
BRASILIA,DF,2131,354216.78
CURITIBA,PR,1521,247392.48


In [0]:
%sql
SELECT 
    estado,
    SUM(quantidade_vendas) AS total_vendas,
    CAST(SUM(valor_total_localidade) AS DECIMAL(12,2)) AS valor_total_estado
FROM medalhao.gold.view_total_compras_por_consumidor
GROUP BY estado
ORDER BY valor_total_estado DESC;


estado,total_vendas,valor_total_estado
SP,41746,5998226.96
RJ,12852,2144379.69
MG,11635,1872257.26
RS,5466,890898.54
PR,5045,811156.38
SC,3637,623086.43
BA,3380,616645.82
DF,2140,355141.08
GO,2020,350092.31
ES,2033,325967.55


## 2. Projeto Logística: Monitoramento de Atrasos

Com o aumento nos índices de atraso, o objetivo é correlacionar regiões e vendedores com problemas de entrega.

#### gold.ft_atrasos_pedidos_local_vendedor
Integração entre Pedidos, Itens (para identificar o vendedor) e Consumidores. Esta tabela serve como base para calcular KPIs de pontualidade.

#### Views de Performance
* **view_tempo_medio_entrega_localidade**: Calcula a média de dias de entrega (real vs. estimado) por cidade.
* **view_vendedor_pontualidade**: Calcula o percentual de pedidos atrasados por vendedor, permitindo identificar gargalos na cadeia de fornecimento.

In [0]:
df_pedidos_total = spark.read.table("medalhao.silver.ft_pedidos")
df_consumidores = spark.read.table("medalhao.silver.ft_consumidores")
df_itens_pedidos = spark.read.table("medalhao.silver.ft_itens_pedidos")

df_informacoes = df_pedidos_total.alias("p").join(
    df_consumidores.alias("c"),
    "id_consumidor", # remove a duplicação da coluna id_consumidor
    "inner"
).join(
    df_itens_pedidos.alias("i"),
    "id_pedido",     #  remove a duplicação da coluna id_pedido
    "inner"
)

df_final = df_informacoes.select(
    F.col("p.id_pedido"),              
    F.col("i.id_vendedor"),          
    F.col("p.id_consumidor"),         
    F.col("p.entrega_no_prazo"), 
    F.col("p.tempo_entrega_dias"),     
    F.col("p.tempo_entrega_estimado_dias"), 
    F.col("c.cidade"),
    F.col("c.estado")
)

df_final.show(5)

salvar_tabela_gold(df_final,"ft_atrasos_pedidos_local_vendedor")


+--------------------+--------------------+--------------------+----------------+------------------+---------------------------+--------------+------+
|           id_pedido|         id_vendedor|       id_consumidor|entrega_no_prazo|tempo_entrega_dias|tempo_entrega_estimado_dias|        cidade|estado|
+--------------------+--------------------+--------------------+----------------+------------------+---------------------------+--------------+------+
|ccbabeb0b02433bd0...|16090f2ca825584b5...|c77ee2d8ba1614a4d...|             Sim|                18|                         22|       UBERABA|    MG|
|c6bf92017bd40729c...|36a968b544695394e...|3d3c463710ea6e8dd...|             Sim|                 7|                          8|        SUMARE|    SP|
|ab87dc5a5f1856a10...|b410bdd36d5db7a65...|538a4d02876412846...|             Sim|                 7|                         43|       UBERABA|    MG|
|06ff862a85c2402aa...|7e3f87d16fb353f40...|0a978c825ff7d0131...|             Sim|             

In [0]:
%sql
CREATE OR REPLACE VIEW medalhao.gold.view_tempo_medio_entrega AS
WITH base AS (
    SELECT
        cidade,
        estado,
        tempo_entrega_dias,
        tempo_entrega_estimado_dias
    FROM medalhao.gold.ft_atrasos_pedidos_local_vendedor
    WHERE entrega_no_prazo IN ('Sim', 'Não')
)

SELECT
    cidade,
    estado,
    CAST(AVG(tempo_entrega_dias) AS DECIMAL(12,2)) AS tempo_medio_entrega,
    CAST(AVG(tempo_entrega_estimado_dias) AS DECIMAL(12,2)) AS tempo_medio_estimado,
    CASE 
        WHEN AVG(tempo_entrega_dias) > AVG(tempo_entrega_estimado_dias)
            THEN 'SIM'
        ELSE 'NÃO'
    END AS entrega_maior_que_estimado
FROM base
GROUP BY cidade, estado;


In [0]:
display(spark.sql("SELECT * FROM medalhao.gold.view_tempo_medio_entrega LIMIT 5"))

cidade,estado,tempo_medio_entrega,tempo_medio_estimado,entrega_maior_que_estimado
SAO PAULO,SP,7.97,18.77,NÃO
BARREIRAS,BA,19.19,30.28,NÃO
VIANOPOLIS,GO,12.25,30.00,NÃO
SAO GONCALO DO AMARANTE,RN,15.50,32.75,NÃO
SANTO ANDRE,SP,7.86,19.44,NÃO


In [0]:
%sql
CREATE OR REPLACE VIEW medalhao.gold.view_pontualidade_pedidos AS
WITH base AS (
    SELECT
        id_vendedor,
        id_pedido,
        entrega_no_prazo
    FROM medalhao.gold.ft_atrasos_pedidos_local_vendedor
    WHERE entrega_no_prazo IN ('Sim', 'Não')
),

agg AS (
    SELECT
        id_vendedor,
        COUNT(DISTINCT id_pedido) AS total_pedidos,
        COUNT(DISTINCT CASE WHEN entrega_no_prazo = 'Não' THEN id_pedido END) AS total_atrasados
    FROM base
    GROUP BY id_vendedor
)

SELECT
    id_vendedor,
    total_pedidos,
    total_atrasados,
    CAST((total_atrasados * 100.0 / total_pedidos) AS DECIMAL(5,2)) AS percentual_atraso
FROM agg
WHERE total_pedidos > 0;


In [0]:
display(spark.sql("SELECT * FROM medalhao.gold.view_pontualidade_pedidos LIMIT 5"))

id_vendedor,total_pedidos,total_atrasados,percentual_atraso
3504c0cb71d7fa48d967e0e4c94d59d9,53,0,0.00
289cdb325fb7e7f891c38608bf9e0962,109,2,1.83
4869f7a5dfa277a7dca6462dcf3b52b2,1124,118,10.50
66922902710d126a0e7d26b0e3805106,150,11,7.33
2c9e548be18521d1c43cde1c582c6de8,124,19,15.32


## 3. Projeto Comercial: Análise Temporal

Para permitir análises de sazonalidade (ex: vendas por dia da semana ou mês), precisamos de uma modelagem dimensional robusta.

#### gold.dm_tempo
Utilizou-se as funções sequence e explode do Spark para gerar um calendário dinâmico entre datas de início e fim.

#### gold.ft_vendas_geral
Esta é a tabela central do modelo ("Big Table"). Ela integra pedidos, itens, clientes, produtos, vendedores e avaliações em uma única estrutura.

#### gold.view_vendas_por_periodo
Agrega métricas como receita_total, total_pedidos e ticket_medio nas granularidades de Ano, Mês e Dia da Semana.

#### gold.view_top_produto
Identifica quais produtos geram mais receita e possuem melhores avaliações, auxiliando na gestão de estoque e marketing.

In [0]:
df_pedidos = spark.table(f"{catalogo}.{silver_db}.ft_pedidos")
#maximo e minimo para intervalo
min_data = df_pedidos.select(F.min(F.to_date("pedido_compra_timestamp"))).first()[0]
max_data = df_pedidos.select(F.max(F.to_date("pedido_compra_timestamp"))).first()[0]

print(f"Intervalo do calendário: {min_data} → {max_data}")

#todas as datas do intervalo
df_tempo = (
    spark.createDataFrame([(1,)], ["dummy"])
    .select(F.explode(F.sequence(F.lit(min_data), F.lit(max_data), F.expr("interval 1 day"))).alias("sk_tempo"))
)

df_tempo = (
    df_tempo
    .withColumn("ano", F.year("sk_tempo"))
    .withColumn("trimestre", F.quarter("sk_tempo"))
    .withColumn("mes", F.month("sk_tempo"))
    .withColumn("semana_do_ano", F.weekofyear("sk_tempo"))
    .withColumn("dia", F.dayofmonth("sk_tempo"))
)


#domingo = 1, sabado = 7
df_tempo = df_tempo.withColumn("dia_da_semana_num", F.dayofweek("sk_tempo"))


df_tempo = df_tempo.withColumn(
    "dia_da_semana_nome",
    F.when(F.col("dia_da_semana_num") == 1, "Domingo")
     .when(F.col("dia_da_semana_num") == 2, "Segunda-feira")
     .when(F.col("dia_da_semana_num") == 3, "Terça-feira")
     .when(F.col("dia_da_semana_num") == 4, "Quarta-feira")
     .when(F.col("dia_da_semana_num") == 5, "Quinta-feira")
     .when(F.col("dia_da_semana_num") == 6, "Sexta-feira")
     .when(F.col("dia_da_semana_num") == 7, "Sábado")
)

df_tempo = df_tempo.withColumn(
    "mes_nome",
    F.when(F.col("mes") == 1, "Janeiro")
     .when(F.col("mes") == 2, "Fevereiro")
     .when(F.col("mes") == 3, "Março")
     .when(F.col("mes") == 4, "Abril")
     .when(F.col("mes") == 5, "Maio")
     .when(F.col("mes") == 6, "Junho")
     .when(F.col("mes") == 7, "Julho")
     .when(F.col("mes") == 8, "Agosto")
     .when(F.col("mes") == 9, "Setembro")
     .when(F.col("mes") == 10, "Outubro")
     .when(F.col("mes") == 11, "Novembro")
     .when(F.col("mes") == 12, "Dezembro")
)

df_tempo = df_tempo.withColumn(
    "eh_fim_de_semana",
    F.when(F.col("dia_da_semana_num").isin(1, 7), "Sim").otherwise("Não")
)

salvar_tabela_gold(df_tempo, "dm_tempo")

display(df_tempo.limit(10))


Intervalo do calendário: 2016-09-04 → 2018-10-17
Tabela gold.dm_tempo criada com sucesso!



sk_tempo,ano,trimestre,mes,semana_do_ano,dia,dia_da_semana_num,dia_da_semana_nome,mes_nome,eh_fim_de_semana
2016-09-04,2016,3,9,35,4,1,Domingo,Setembro,Sim
2016-09-05,2016,3,9,36,5,2,Segunda-feira,Setembro,Não
2016-09-06,2016,3,9,36,6,3,Terça-feira,Setembro,Não
2016-09-07,2016,3,9,36,7,4,Quarta-feira,Setembro,Não
2016-09-08,2016,3,9,36,8,5,Quinta-feira,Setembro,Não
2016-09-09,2016,3,9,36,9,6,Sexta-feira,Setembro,Não
2016-09-10,2016,3,9,36,10,7,Sábado,Setembro,Sim
2016-09-11,2016,3,9,36,11,1,Domingo,Setembro,Sim
2016-09-12,2016,3,9,37,12,2,Segunda-feira,Setembro,Não
2016-09-13,2016,3,9,37,13,3,Terça-feira,Setembro,Não


In [0]:
df_pedidos = spark.table(f"{catalogo}.{silver_db}.ft_pedidos")
df_itens = spark.table(f"{catalogo}.{silver_db}.ft_itens_pedidos")
df_avaliacoes = spark.table(f"{catalogo}.{silver_db}.ft_avaliacoes_pedidos")
df_dolar = spark.table(f"{catalogo}.{silver_db}.dm_cotacao_dolar")
df_produtos = spark.table(f"{catalogo}.{silver_db}.ft_produtos")

df_item_valores = (
    df_itens
    .withColumn("valor_total_item_brl", 
                F.col("preco_BRL") + F.col("preco_frete"))
)

df_base = (
    df_item_valores.alias("i")
    .join(df_pedidos.alias("p"), F.col("i.id_pedido") == F.col("p.id_pedido"), "inner")
)

df_base = df_base.join(
    df_dolar.alias("d"),
    F.to_date(F.col("p.pedido_compra_timestamp")) == F.col("d.data"),
    "left"
)


df_avaliacoes_agg = (
    df_avaliacoes
    .groupBy("id_pedido")
    .agg(F.avg("avaliacao").alias("avaliacao_pedido"))
)

df_base = df_base.join(df_avaliacoes_agg, "id_pedido", "left")

df_base = (
    df_base
    .withColumn("valor_produto_usd", F.round(F.col("i.preco_BRL") / F.col("d.cotacao_dolar"), 2))
    .withColumn("valor_frete_usd", F.round(F.col("i.preco_frete") / F.col("d.cotacao_dolar"), 2))
    .withColumn("valor_total_item_usd", F.round(F.col("valor_total_item_brl") / F.col("d.cotacao_dolar"), 2))
)
df_base = (
    df_base
    .withColumn("valor_produto_usd", F.round(F.col("i.preco_BRL") / F.col("d.cotacao_dolar"), 2))
    .withColumn("valor_frete_usd", F.round(F.col("i.preco_frete") / F.col("d.cotacao_dolar"), 2))
    .withColumn("valor_total_item_usd", F.round(F.col("valor_total_item_brl") / F.col("d.cotacao_dolar"), 2))
)


df_vendas_geral = (
    df_base.select(
        F.col("p.id_pedido"),
        F.col("i.id_item"),
        F.col("p.id_consumidor").alias("fk_cliente"),
        F.col("i.id_produto").alias("fk_produto"),
        F.col("i.id_vendedor").alias("fk_vendedor"),
        F.to_date("p.pedido_compra_timestamp").alias("fk_tempo"),
        F.col("p.status").alias("status_pedido"),
        F.col("p.tempo_entrega_dias"),
        F.col("p.entrega_no_prazo"),
        F.col("i.preco_BRL").alias("valor_produto_brl"),
        F.col("i.preco_frete").alias("valor_frete_brl"),
        F.col("valor_total_item_brl"),
        F.col("valor_produto_usd"),
        F.col("valor_frete_usd"),
        F.col("valor_total_item_usd"),
        F.col("d.cotacao_dolar"),
        F.col("avaliacao_pedido")
    )
)

display(df_vendas_geral.limit(5))
salvar_tabela_gold(df_vendas_geral, "ft_vendas_geral")


id_pedido,id_item,fk_cliente,fk_produto,fk_vendedor,fk_tempo,status_pedido,tempo_entrega_dias,entrega_no_prazo,valor_produto_brl,valor_frete_brl,valor_total_item_brl,valor_produto_usd,valor_frete_usd,valor_total_item_usd,cotacao_dolar,avaliacao_pedido
019886de8f385a39b75bedbb726fd4ef,1,8cf88d7ba142365ef2ca619ef06f9a0f,e9a69340883a438c3f91739d14d3a56d,1b4c3a6f53068f0b6944d2d005c9fc89,2018-02-10,entregue,13,Sim,159.9,28.5,188.4,48.73,8.69,57.41,3.2815,5.0
06ff862a85c2402aa52dc9edf150bf30,1,0a978c825ff7d013133ddc7f77566172,4ce9ab528124f89e091b17d11aa2e97c,7e3f87d16fb353f408d467e74fbd8014,2017-11-30,entregue,28,Sim,41.9,17.63,59.53,12.85,5.41,18.26,3.261,3.0
110ac0768c3e3a78e2937b1cb1ea3395,1,770d7945efb2aa13cbcf28eb7a353def,1b7ce992a80ac036dd9ab73d08289712,582d4f8675b945722eda7c0cb61ba4c7,2017-06-09,entregue,5,Sim,25.5,15.1,40.6,7.79,4.61,12.4,3.2734,5.0
12dd6fe47b1a7c9e5742662223880dcc,1,95a911adbd9c6dee5b6ba9f764678fa3,63f4d40c05db6ade462cecef857eec34,fec6275253471ace26d209bbaa64cd0f,2018-01-24,entregue,14,Sim,284.0,39.54,323.54,88.85,12.37,101.22,3.1964,5.0
139be8870b91e71fd70bb366305c8cde,1,7f0f4bcffd7d9085eed3ad0a7814dd33,4c1e109ecdf58453de365d217cefa64c,4e922959ae960d389249c378d1c939f5,2017-05-13,entregue,9,Sim,120.0,13.18,133.18,38.36,4.21,42.57,3.1284,5.0


Tabela gold.ft_vendas_geral criada com sucesso!



In [0]:
%sql
CREATE OR REPLACE VIEW medalhao.gold.view_vendas_por_periodo AS
SELECT
    t.ano,
    t.trimestre,
    t.mes,
    t.mes_nome,
    t.dia,
    t.dia_da_semana_num,

    COUNT(DISTINCT f.id_pedido) AS total_pedidos,
    COUNT(f.id_item) AS total_itens,

    ROUND(SUM(f.valor_total_item_brl), 2) AS receita_total_brl,
    ROUND(SUM(f.valor_total_item_usd), 2) AS receita_total_usd,

    ROUND(AVG(f.valor_total_item_brl), 2) AS ticket_medio_brl,
    ROUND(AVG(f.avaliacao_pedido), 2) AS avaliacao_media

FROM medalhao.gold.ft_vendas_geral f
JOIN medalhao.gold.dm_tempo t
    ON f.fk_tempo = t.sk_tempo

GROUP BY 
    t.ano,
    t.trimestre,
    t.mes,
    t.mes_nome,
    t.dia,
    t.dia_da_semana_num
;


In [0]:
display(spark.sql("SELECT * FROM medalhao.gold.view_vendas_por_periodo LIMIT 5"))


ano,trimestre,mes,mes_nome,dia,dia_da_semana_num,total_pedidos,total_itens,receita_total_brl,receita_total_usd,ticket_medio_brl,avaliacao_media
2017,1,1,Janeiro,10,3,6,6,1571.17,492.36,261.86,3.5
2017,1,2,Fevereiro,7,3,111,127,16524.21,5279.64,130.11,4.26
2018,2,6,Junho,18,2,246,279,41023.4,10928.85,147.04,4.51
2017,2,6,Junho,17,7,74,84,12746.15,3875.87,151.74,3.95
2018,2,4,Abril,18,4,280,315,46295.73,13681.68,146.97,4.01


In [0]:
%sql
SELECT dia_da_semana_num, ROUND(SUM(receita_total_brl),2) AS receita
FROM medalhao.gold.view_vendas_por_periodo
GROUP BY dia_da_semana_num
ORDER BY receita DESC
LIMIT 1;


dia_da_semana_num,receita
2,2600533.82


Logo o dia é segunda!

In [0]:
%sql
WITH ult_ano AS (
    SELECT MAX(ano) AS ano_ref FROM medalhao.gold.dm_tempo
)
SELECT v.mes, v.mes_nome, ROUND(AVG(v.ticket_medio_brl),2) AS ticket_medio
FROM medalhao.gold.view_vendas_por_periodo v
JOIN ult_ano u ON v.ano = u.ano_ref
GROUP BY v.mes, v.mes_nome
ORDER BY ticket_medio DESC
LIMIT 1;


mes,mes_nome,ticket_medio
9,Setembro,166.46


In [0]:
%sql
CREATE OR REPLACE VIEW medalhao.gold.view_top_produto AS
SELECT
    f.fk_produto AS id_produto,
    p.categoria_produto,

    COUNT(f.id_item) AS quantidade_vendida,
    COUNT(DISTINCT f.id_pedido) AS total_pedidos,

    ROUND(SUM(f.valor_produto_brl), 2) AS receita_brl,
    ROUND(SUM(f.valor_produto_usd), 2) AS receita_usd,

    ROUND(AVG(f.valor_produto_brl), 2) AS preco_medio_brl,
    ROUND(AVG(f.avaliacao_pedido), 2) AS avaliacao_media,

    ROUND(AVG(p.peso_produto_gramas), 2) AS peso_medio_gramas

FROM medalhao.gold.ft_vendas_geral f
JOIN medalhao.silver.ft_produtos p
    ON f.fk_produto = p.id_produto

GROUP BY 
    f.fk_produto,
    p.categoria_produto;


In [0]:
display(spark.sql("SELECT * FROM medalhao.gold.view_top_produto LIMIT 5"))


id_produto,categoria_produto,quantidade_vendida,total_pedidos,receita_brl,receita_usd,preco_medio_brl,avaliacao_media,peso_medio_gramas
2b4609f8948be18874494203496bc318,beleza_saude,260,259,22717.22,6886.36,87.37,4.06,250.0
675eadffd43ac06d1bead694fb553d35,moveis_decoracao,1,1,109.9,35.3,109.9,5.0,2000.0
8d063218dab590d1a369a08e761c16e1,beleza_saude,1,1,31.99,10.03,31.99,5.0,388.0
54f45413afc0d5480c59f5dfeed2d19f,beleza_saude,9,7,1199.1,348.61,133.23,3.33,2450.0
9239c49d7d0e542d0e12a9023771ef51,construcao_ferramentas_jardim,1,1,29.45,9.22,29.45,5.0,155.0


## 4. Análise de Produtos e Categoria Fashion

Foco na performance individual dos produtos e na categoria estratégica de "Fashion".

#### gold.view_vendas_produtos_esteticos
Esta view utiliza **CTE (Common Table Expression)** para filtrar primeiramente os dados brutos e depois agregar as métricas de vendas e avaliações mensais apenas para categorias que iniciam com "fashion".

In [0]:
%sql
CREATE OR REPLACE VIEW medalhao.gold.view_vendas_produtos_esteticos AS
WITH base AS (
    SELECT
        f.id_pedido,
        f.id_item,
        f.valor_total_item_brl,
        f.valor_total_item_usd,
        f.avaliacao_pedido,
        p.categoria_produto,
        t.ano,
        t.mes
    FROM medalhao.gold.ft_vendas_geral f
    JOIN medalhao.silver.ft_produtos p ON f.fk_produto = p.id_produto
    JOIN medalhao.gold.dm_tempo t ON t.sk_tempo = f.fk_tempo
    WHERE p.categoria_produto LIKE 'fashion%'
)
SELECT
    ano,
    mes,
    categoria_produto,
    COUNT(DISTINCT id_pedido) AS total_pedidos,
    COUNT(id_item) AS total_itens_vendidos,
    SUM(valor_total_item_brl) AS receita_total_brl,
    SUM(valor_total_item_usd) AS receita_total_usd,
    AVG(valor_total_item_brl) AS ticket_medio_brl,
    AVG(valor_total_item_usd) AS ticket_medio_usd,
    AVG(avaliacao_pedido) AS avaliacao_media
FROM base
GROUP BY ano, mes, categoria_produto
ORDER BY ano, mes;


In [0]:
display(spark.sql("SELECT * FROM medalhao.gold.view_vendas_produtos_esteticos LIMIT 20"))

ano,mes,categoria_produto,total_pedidos,total_itens_vendidos,receita_total_brl,receita_total_usd,ticket_medio_brl,ticket_medio_usd,avaliacao_media
2016,10,fashion_roupa_masculina,1,1,35.86,null,35.86,null,1.0
2016,10,fashion_calcados,1,1,40.95,null,40.95,null,5.0
2016,10,fashion_roupa_feminina,1,1,69.63,null,69.63,null,5.0
2016,10,fashion_bolsas_e_acessorios,8,9,656.4900000000001,null,72.94333333333334,null,3.2222222222222223
2016,12,fashion_bolsas_e_acessorios,1,1,19.62,null,19.62,null,5.0
2017,1,fashion_calcados,1,1,49.01,15.36,49.01,15.36,5.0
2017,1,fashion_bolsas_e_acessorios,30,35,2136.609999999999,671.8,61.04599999999998,19.194285714285712,4.363636363636363
2017,1,fashion_roupa_feminina,2,2,102.81,32.46,51.405,16.23,4.0
2017,1,fashion_underwear_e_moda_praia,2,2,171.15,54.05,85.575,27.025,3.0
2017,1,fashion_roupa_masculina,1,1,159.11,49.86,159.11,49.86,5.0


Atenção: primeiros valores nulos para usd, pois não temos cotações disponíveis, vale lembrar que a primeira cotação é de 2017-01-02, uma estratégia seria usarmos essa cotação para esses valores!